# Notebook 1: Real-Time Acoustic Kinematics & Telemetry Dashboard

Welcome to **`pynq-sound-localizer`** (`v1.0.0`). This notebook demonstrates how to load the FPGA hardware overlay, launch the **10-Second Rolling Kinematics Dashboard**, record continuous multi-second flight data, and extract clean Doppler telemetry.

## 1. Load the Hardware Overlay
Instantiate `MicrophoneArrayOverlay()`. It automatically downloads and configures the verified `v1.5.0` bitstream on the PYNQ-Z2 board.

In [ ]:
from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

# Auto-detect board and load overlay
ol = MicrophoneArrayOverlay()
print(f"✅ Overlay loaded: {ol.current_profile} mode ({ol.fs_per_ch:.0f} SPS per channel)")
print(f"   Speed of sound at 20°C: {KinematicAnalytics.speed_of_sound(20.0):.2f} m/s")

## 2. Launch the Real-Time 10-Second Rolling Telemetry Dashboard
Click **`Start Stream`** and make sound near the microphones (or wave a pure tone generator from a smartphone). Observe:
- **Row 1:** Instantaneous physical loudness envelope $A(t)$ in Volts.
- **Row 2:** Sub-Hertz dominant pitch scatter points $f_0(t)$ with real-time noise squelching.
- **Tabs:** Switch seamlessly between **Mic 1 (A0)**, **Mic 2 (A1)**, and **Dual Overlay**.

In [ ]:
# Launch the multi-tab live instrument
app = ol.kinematics_dashboard()

## 3. Direct Clean Data Handoff in Python
Extract the sanitized non-NaN data directly from the dashboard into NumPy arrays for instant analysis.

In [ ]:
t_clean, amp_clean, freq_clean = app.get_clean_data(channel=1)
print(f"Captured {len(t_clean)} clean motion points!")
if len(freq_clean) > 0:
    print(f"Frequency span: {freq_clean.min():.1f} Hz -> {freq_clean.max():.1f} Hz (Shift: {freq_clean.max() - freq_clean.min():.1f} Hz)")

## 4. Continuous Multi-Second Flight Recording & Audio Playback
Record uninterrupted audio to DDR memory and listen to it directly in Jupyter.

In [ ]:
# Record 3.0 seconds of 50 kSPS dual-channel audio
t_axis, v_mic1, v_mic2 = ol.record_continuous(duration_sec=3.0)

# Listen to Microphone 1
ol.play_audio(channel=1, custom_data=v_mic1)

## 5. Hardware Shutdown & Cleanup

In [ ]:
app.stop()
ol.close()
print("🔒 Hardware resources cleanly released.")